# Predict teacher talk moves

In [1]:
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

from edubehaviors import AssertionAnnotator, ClassificationPipeline, WordAnnotator, standard_classifier

## Setup data

In [2]:
random_state = 2026_09_17

In [3]:
# Load data and set outcome column
data = pd.read_csv("talkmoves_tutor.csv")
OUTCOME = "label_press_for_accuracy"

## Predict in one step using `ClassificationPipeline`

In [4]:
pipeline = ClassificationPipeline(
    data,
    words="all",
    assertions=["sentence_has_a_question", "sentence_has_math_terms", "sentence_calls_on_student_by_name"],
    label_column=OUTCOME,
    text_column="sentence",
    random_state=random_state,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [5]:
print(pipeline.report())

              precision    recall  f1-score   support

       False       0.95      0.85      0.90       291
        True       0.51      0.78      0.62        59

    accuracy                           0.84       350
   macro avg       0.73      0.81      0.76       350
weighted avg       0.88      0.84      0.85       350



## Predict manually with Annotators and `standard_classifier`

In [6]:
# Create word features
wordannotator = WordAnnotator()
word_features = wordannotator.annotate(data.sentence)

In [7]:
# create assertion features
assertionannotator = AssertionAnnotator(
    ["sentence_has_a_question", "sentence_has_math_terms", "sentence_calls_on_student_by_name"]
)
assertion_features = assertionannotator.annotate(data.sentence)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [8]:
# Combine features
all_features = pd.concat([word_features, assertion_features], axis=1)

In [9]:
# Check outcome counts
outcome = data[OUTCOME]
outcome.value_counts()

label_press_for_accuracy
False    1455
True      292
Name: count, dtype: int64

In [10]:
# Split into train and test set
train_x, test_x, train_y, test_y = train_test_split(
    all_features, outcome, test_size=0.2, stratify=outcome, shuffle=True, random_state=random_state
)

In [11]:
# Create and fit classifier
classifier = standard_classifier(random_state=random_state)
classifier.fit(train_x, train_y)

,Cs,"(0.001, ...)"
,fit_intercept,True
,cv,5
,dual,False
,penalty,'l2'
,scoring,'f1_macro'
,solver,'lbfgs'
,tol,0.0001
,max_iter,100
,class_weight,'balanced'
,n_jobs,None


In [12]:
# Predict test set
pred_y = classifier.predict(test_x)

In [13]:
# Get evaluation metrics
print(classification_report(test_y, pred_y))

              precision    recall  f1-score   support

       False       0.94      0.85      0.89       291
        True       0.49      0.71      0.58        59

    accuracy                           0.83       350
   macro avg       0.71      0.78      0.73       350
weighted avg       0.86      0.83      0.84       350

